# Tirex Evaluation

In [1]:
import sys
sys.path.append('../tirex/src') # Add the path to the tirex module

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
from datetime import datetime
from pathlib import Path

from tirex import ForecastModel, load_model
from tirex.util import plot_forecast

# set default figure size for all plots
plt.rcParams["figure.figsize"] = (12, 6)
# set default seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Load data

Download the data from the notebook [data_download.ipynb](data_download.ipynb) before.

## Fine-Tuning Data

In [3]:
data_path_fets = "../data/data_fets.csv"
data_fets = pl.read_csv(data_path_fets, try_parse_dates=True)
data_fets

timestamp,value,series_index
"datetime[μs, UTC]",f64,i64
2019-12-31 23:00:00 UTC,43881.8,0
2019-12-31 23:15:00 UTC,43639.6,0
2019-12-31 23:30:00 UTC,43330.9,0
2019-12-31 23:45:00 UTC,43149.5,0
2020-01-01 00:00:00 UTC,43017.3,0
…,…,…
2025-10-13 20:45:00 UTC,51524.0,0
2025-10-13 21:00:00 UTC,50427.2,0
2025-10-13 21:15:00 UTC,49845.8,0


# Load Model

In [4]:
model: ForecastModel = load_model("NX-AI/TiRex")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Model device: {device}")

Total Parameters: 35,291,200
Model device: cuda


In [5]:
# for restoring the original behavior of the model after our modifications
from tirex import TiRexZero

TiRexZero._original_forecast_tensor = TiRexZero._forecast_tensor

def restore_original_behavior(print_message: bool = True):
    if hasattr(TiRexZero, '_original_forecast_tensor'):
        TiRexZero._forecast_tensor = TiRexZero._original_forecast_tensor
        if print_message:
            print("Restored original NaN-filling behavior.")
    else:
        print("Original method backup not found.")

# Dataloader

Dataloader produces sequences of length $n$ for the input and target sequences with random predictions lengths in the range of $s_{\min}$ and $s_{\max}$.

In [6]:
from torch.utils.data import DataLoader
from tirex_loss.dataloader import build_dataloader, build_dataloader_from_dataset, TirexDataset
from tirex_loss.dataloader.utils import create_windows

In [ ]:
n = model.config.train_ctx_len
s_min = 32
s_max = 128
batch_size = 1 # use batch size 1, sklearn metrics need input (n_samples,)
train_ratio = 0.8

data_fets_filtered = data_fets
seq, lengths = create_windows(data_fets_filtered, n=n, s_min=s_min, s_max=s_max)

idx = np.random.permutation(len(seq))
seq, lengths = seq[idx], lengths[idx]

split = int(len(seq) * train_ratio)
seq_train, seq_test = seq[:split], seq[split:]
lengths_train, lengths_test = lengths[:split], lengths[split:]

dataset_train = TirexDataset(sequences=seq_train,
                             prediction_lengths=lengths_train,
                             context_length=n,
                             prediction_length_min=s_min,
                             prediction_length_max=s_max
                             )
dataset_test = TirexDataset(sequences=seq_test,
                            prediction_lengths=lengths_test,
                            context_length=n,
                            prediction_length_min=s_min,
                            prediction_length_max=s_max
                            )

dataloader_train = build_dataloader_from_dataset(dataset_train,
                                    batch_size=batch_size,
                                    shuffle=True,
                                    num_workers=0,
                                    pin_memory=True
                                    )
dataloader_test = build_dataloader_from_dataset(dataset_test,
                                   batch_size=batch_size,
                                   shuffle=False,
                                   num_workers=0,
                                   pin_memory=True
                                   )

len(dataloader_train), len(dataloader_test)

(250, 61)

# Evaluation

In [8]:
from tqdm.auto import tqdm
from tirex_loss.modifications.autoregressive import autoregressive_mean_forecast_tensor
from tirex_loss.evaluation.metrics import score_data

model_files = ['best_model.pth', 'last_model.pth']

In [ ]:
def compare_autoregressive(dataloader_test: DataLoader, device:str):
    scores_comparison = {"original": (0, 0, 0, 0), "autoregressive_mean": (0, 0, 0, 0)}
    quantiles_comparison = {"original": [], "autoregressive_mean": []}
    mean_comparison = {"original": [], "autoregressive_mean": []}

    batch_bar = tqdm(dataloader_test, total=len(dataloader_test),
                            leave=False, position=1, desc='Batches')
    for i, (x_batch, y_batch, pred_length_batch) in enumerate(batch_bar):
        # put tensors on same device
        x = x_batch.to(device)
        y = y_batch.to(device)
        pred_length = pred_length_batch[0].item()

        # change shape to (n_samples, n_outputs)
        y_true = y.reshape(-1, 1)
        y_train = x.reshape(-1, 1)

        restore_original_behavior(print_message=False)
        quantiles_original, mean_original = model.forecast(x,
                                                           prediction_length=pred_length,
                                                           output_device=device
                                                           )
        quantiles_comparison["original"].append(quantiles_original.cpu().numpy())
        mean_comparison["original"].append(mean_original.cpu().numpy())

        # scores
        y_pred = quantiles_original.reshape(quantiles_original.shape[0]*quantiles_original.shape[1],1,-1)
        score_mape, score_mase, score_r2, score_wis = score_data(
                    y_true=y_true.cpu().numpy(),
                    y_pred=y_pred.cpu().numpy(),
                    y_train=y_train.cpu().numpy(),
                    quantiles=model.config.quantiles,
                    log_scores=False,
                    multioutput='uniform_average'
                    )

        scores_comparison["original"] = tuple(a + b for a, b in zip(scores_comparison["original"], (score_mape, score_mase, score_r2, score_wis)))

        # now with autoregressive mean filling
        TiRexZero._forecast_tensor = autoregressive_mean_forecast_tensor
        quantiles_autoregressive, mean_autoregressive = model.forecast(x,
                                                                       prediction_length=pred_length,
                                                                       output_device=device
                                                                       )
        quantiles_comparison["autoregressive_mean"].append(quantiles_autoregressive.cpu().numpy())
        mean_comparison["autoregressive_mean"].append(mean_autoregressive.cpu().numpy())

        # scores
        y_pred = quantiles_autoregressive.reshape(quantiles_autoregressive.shape[0]*quantiles_autoregressive.shape[1],1,-1)
        score_mape, score_mase, score_r2, score_wis = score_data(
                    y_true=y_true.cpu().numpy(),
                    y_pred=y_pred.cpu().numpy(),
                    y_train=y_train.cpu().numpy(),
                    quantiles=model.config.quantiles,
                    log_scores=False,
                    multioutput='uniform_average'
                    )

        scores_comparison["autoregressive_mean"] = tuple(a + b for a, b in zip(scores_comparison["autoregressive_mean"], (score_mape, score_mase, score_r2, score_wis)))

    scores_comparison["original"] = tuple(score / len(dataloader_test) for score in scores_comparison["original"])
    scores_comparison["autoregressive_mean"] = tuple(score / len(dataloader_test) for score in scores_comparison["autoregressive_mean"])    

    return scores_comparison, quantiles_comparison, mean_comparison

## Original training behavior

In [14]:
# load model
model_path = Path("../models/tirex_original_20260516_232755")

scores = {}
quantiles = {}
for model_file in model_files:
    dict_path = model_path / model_file
    state_dict = torch.load(dict_path, weights_only=True)
    model.load_state_dict(state_dict)

    scores_comparison, quantiles_comparison, mean_comparison = compare_autoregressive(dataloader_test, device)
    scores[model_file] = scores_comparison
    quantiles[model_file] = quantiles_comparison

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

ValueError: y_true and y_pred must have same shape, got (1024, 1) vs (128, 8)

## Autoregressive training behavior

In [12]:
# load model
model_path = Path("../models/tirex_autoregressive_20260517_221701")

scores_autoregressive = {}
quantiles_autoregressive = {}
for model_file in model_files:
    dict_path = model_path / model_file
    state_dict = torch.load(dict_path, weights_only=True)
    model.load_state_dict(state_dict)

    scores_comparison, quantiles_comparison, mean_comparison = compare_autoregressive(dataloader_test, device)
    scores_autoregressive[model_file] = scores_comparison
    quantiles_autoregressive[model_file] = quantiles_comparison

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

c:\py_venv\pw_tirex\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
c:\py_venv\pw_tirex\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


Batches:   0%|          | 0/61 [00:00<?, ?it/s]

c:\py_venv\pw_tirex\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
c:\py_venv\pw_tirex\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
